# 02 코드 해설 · GPU에서 모델을 조금 더 학습시키기

이 노트북은 [02_gpu_finetuning.ipynb](../notebooks/02_gpu_finetuning.ipynb)의 코드를 처음 읽는 학생을 위한 해설입니다. **원본의 코드 셀 24개를 빠짐없이 담고**, 두 빈칸 실습은 문제의 의미와 확인 방법을 설명합니다. 완성 답안을 대신 제출하는 노트북은 아닙니다.

원본 GPU 코드는 아래 회색 코드 블록에 **읽기용 Markdown**으로 들어 있습니다. 여기에 있는 “모두 실행”은 GPU 연결이나 학습을 시작하지 않습니다. 실행 가능한 셀은 “작은 CPU 연습”으로 표시한 표준 Python 예제 7개뿐이며, 라이브러리 설치·인터넷·파일 읽기·쓰기를 하지 않습니다. Python 3 커널로 열면 됩니다.

**읽는 순서:** 셀의 목적을 읽고 원본 코드를 살펴본 뒤 예상 결과와 오류 설명으로 돌아오세요. 실제 학습은 00번의 준비와 01번 추론을 마친 후 원본 02번에서 진행합니다. 작은 연습의 가상 숫자는 실제 GPU 성능 결과와 구분합니다.

| 구간 | 이해할 질문 |
|---|---|
| 준비와 데이터 | 어떤 파일을 읽고, 왜 GPU·정답·중복을 확인할까? |
| 헤드 학습 | 왜 965개만 학습할까? 손실과 가중치 업데이트는 어떻게 다를까? |
| 마지막 블록 학습 | 학습 범위가 바뀌면 무엇이 달라질까? |
| 평가와 저장 | 어떤 시점의 모델을 고르고, 저장한 결과를 어떻게 확인할까? |

이 해설은 저장소 기준 커밋 `46d05fe733990c52e47b6882413b7f4181c859c1`의 원본을 따라갑니다. 아래 “원본 셀 번호”는 Markdown까지 포함해 **1부터 센 전체 위치**이며, `In [ ]`에 표시되는 실행 번호와 다릅니다. 파일 안의 추적용 메타데이터만 0부터 센 번호를 사용합니다. 이후 원본 코드가 바뀌면 해설도 다시 확인해야 합니다.

## 01. 이전 실행의 흔적을 지우고 시작하기

**원본 셀 번호:** `2`

**원본 코드 — 읽기 전용**

```python
TRAINING_ALLOWED = False
EXERCISE_CHECKS = {"02-step": False, "02-scope": False}
HEAD_STAGE_COMPLETE = False
FINETUNE_SETUP_COMPLETE = False
FINETUNE_STAGE_COMPLETE = False
MODEL_RELOADED = False
TABLES_SAVED = False
REPORT_READY = False
for name in ("train_one_batch", "select_finetune_parameters", "report"):
    globals().pop(name, None)
```

이 셀은 학습을 시작하기 전 확인표를 모두 “아직 안 끝남”으로 돌립니다. 변수는 값을 붙여 두는 이름이며, `False`와 `True`는 각각 거짓과 참입니다.

- `TRAINING_ALLOWED`는 준비를 마쳤는지, `EXERCISE_CHECKS`는 두 빈칸 실습의 검사를 통과했는지 기록합니다. 사전의 `"02-step"`과 `"02-scope"`는 검사 항목의 이름입니다.
- `HEAD_STAGE_COMPLETE`부터 `REPORT_READY`까지는 헤드 학습, 추가 학습, 재로딩, 결과 저장을 순서대로 마쳤는지 기록합니다. 실제 학습과 저장은 뒤쪽 셀에서 합니다.
- `globals()`는 현재 노트북에서 만든 이름들의 공간입니다. `pop(name, None)`은 예전 실행의 함수나 보고서가 남아 있으면 지웁니다. 없을 때는 `None`을 돌려주므로 오류가 나지 않습니다.

**읽고 확인하기:** 빈칸을 지웠는데 예전 함수가 메모리에 남아 있으면 실습을 완성했다고 잘못 판단할 수 있습니다. 이 셀이 그 혼동을 막습니다. 정상 실행해도 출력은 없습니다. 중간에 이 셀만 다시 실행하면 완료 표시도 초기화되므로 원본은 위에서 아래로 진행합니다.

## 02. 필요한 도구를 가져오고 GPU를 확인하기

**원본 셀 번호:** `4`

**원본 코드 — 읽기 전용**

```python
TRAINING_ALLOWED = False
import os
import sys
import json
import hashlib
import random
from pathlib import Path
from importlib.metadata import version

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython import get_ipython
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")
if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. Colab CLI GPU 세션으로 실행하세요.")
device = torch.device("cuda")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("Python:", sys.executable)
print("GPU:", torch.cuda.get_device_name(0))
print({name: version(name) for name in ("torch", "transformers", "huggingface-hub")})
```

`import`는 설치되어 있는 라이브러리를 현재 코드에서 쓰겠다는 선언입니다. 설치 명령과는 다릅니다. 이 셀의 여러 import는 다음 일을 나눠 맡습니다.

| 이름 | 이 노트북에서 하는 일 |
|---|---|
| `os`, `sys`, `Path` | 환경 설정, 실행 중인 Python 위치, 파일 경로 |
| `json`, `hashlib` | 실행 기록 읽기·쓰기, 파일과 가중치 비교 |
| `random`, `np`, `torch` | 난수 설정, 배열 계산, 모델 학습 |
| `Image`, `plt` | 이미지 변환, 결과 그림 |
| `AutoImageProcessor`, `AutoModelForImageClassification` | 모델에 맞는 전처리와 모델 불러오기 |

`HF_HUB_OFFLINE`과 `TRANSFORMERS_OFFLINE`은 준비해 둔 파일을 사용하도록 설정합니다. 파일이 빠졌을 때 자동으로 내려받아 문제를 감추는 대신 오류를 확인할 수 있습니다. `CUBLAS_WORKSPACE_CONFIG`와 결정적 연산 설정은 같은 환경에서 결과를 비교하기 쉽게 돕습니다. 모든 GPU·라이브러리 버전에서 숫자가 완전히 같아진다는 뜻은 아닙니다.

`torch.cuda.is_available()`가 `False`이면 즉시 멈춥니다. `device`는 뒤에서 모델과 입력을 옮길 목적지이며 여기서는 `cuda`, 즉 GPU입니다. `SEED = 42`는 난수의 시작점을 맞추는 설정입니다. 모델의 초기값과 데이터 순서를 비교할 때 도움이 됩니다.

**실행 결과:** Python 경로, GPU 이름, 세 라이브러리 버전이 출력됩니다. GPU 이름은 실제 배정받은 장치에 따라 달라집니다. “GPU가 없습니다”가 나오면 원본은 Codespaces CPU가 아닌 Colab CLI GPU 세션에서 실행해야 합니다. 이 해설 노트북의 작은 연습 셀은 CPU에서 실행해도 됩니다.

## 03. 실습 폴더와 모델 파일이 맞는지 확인하기

**원본 셀 번호:** `5`

**원본 코드 — 읽기 전용**

```python
candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/vision-ai")]
ROOT = next((p for p in candidates if (p / ".vision-lab-root").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(".vision-lab-root와 실습 파일을 먼저 업로드하세요.")
MODEL_ID = 'facebook/deit-tiny-patch16-224'
MODEL_REVISION = 'b3428f18dcc7b543470d07f14b4a4157815d1880'
MODEL_DIR = ROOT / "hf_colab_gpu/models/deit-tiny"
DATA_DIR = ROOT / "data/prepared"
OUTPUT_DIR = ROOT / "hf_colab_gpu/results/gpu"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

download_manifest = json.loads((MODEL_DIR / "download_manifest.json").read_text())
assert download_manifest["model_id"] == MODEL_ID
assert download_manifest["revision"] == MODEL_REVISION
for name in ("config.json", "preprocessor_config.json", "pytorch_model.bin"):
    assert sha256_file(MODEL_DIR / name) == download_manifest["files"][name], name
print("HF CLI 다운로드 파일 확인:", MODEL_ID, MODEL_REVISION[:12])
```

`Path.cwd()`는 지금 작업 중인 폴더입니다. `parents`는 그 상위 폴더들이고, `*`는 그 목록을 펼칩니다. `candidates`에 있는 위치 중 `.vision-lab-root` 표시 파일을 찾은 첫 폴더를 `ROOT`로 삼습니다. 어디서 노트북을 열었는지 달라도 실습 루트를 찾기 위한 코드입니다.

`ROOT / "경로"`에서 `/`는 나눗셈이 아니라 경로를 이어 붙이는 연산입니다. `MODEL_DIR`는 사전학습 모델, `DATA_DIR`는 준비된 이미지, `OUTPUT_DIR`는 이번 결과를 보관할 폴더입니다. `mkdir(parents=True, exist_ok=True)`는 중간 폴더까지 만들고 이미 있는 폴더는 그대로 둡니다. 완료 보고서의 덮어쓰기는 뒤쪽 준비 셀에서 별도로 막습니다.

`MODEL_ID`는 Hub의 모델 이름, `MODEL_REVISION`은 이번 수업에서 고정한 버전 식별자입니다. `sha256_file()`은 파일 내용을 읽고 SHA-256 문자열을 만듭니다. 파일 내용이 달라졌는지 비교하는 표식이라고 생각하면 됩니다.

`download_manifest.json`은 00번에서 받은 모델 이름·버전·파일별 SHA-256 기록입니다. 세 번의 `assert` 검사로 이름, 버전, 실제 파일을 대조합니다. `assert 조건`은 조건이 거짓이면 멈춥니다. 이것은 파일이 기록과 같은지 확인하는 절차이며 모델의 정확도나 안전성을 보장하는 검사는 아닙니다.

**실행 결과:** `HF CLI 다운로드 파일 확인:` 다음에 모델 이름과 버전 앞 12글자가 나옵니다. 파일 누락이나 SHA-256 불일치는 00번에서 준비한 파일의 업로드 상태부터 확인합니다.

## 04. 이미지·정답·분할 기록을 함께 검사하기

**원본 셀 번호:** `7`

**원본 코드 — 읽기 전용**

```python
manifest_file = DATA_DIR / "manifest.json"
manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
classes = manifest["classes"]
if len(classes) < 2 or len(set(classes)) != len(classes):
    raise ValueError("클래스 목록을 확인하세요.")
identity = {k: manifest[k] for k in ("classes", "splits", "seed", "preprocess")}
fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
assert fingerprint == manifest["dataset_sha256"]
splits, seen_ids = {}, set()
for split in ("train", "validation", "test"):
    record = manifest["splits"][split]
    assert record["file"] == f"{split}.npz"
    file = DATA_DIR / record["file"]
    assert sha256_file(file) == record["sha256"], split
    with np.load(file, allow_pickle=False) as stored:
        images, labels, ids = stored["images"], stored["labels"], stored["ids"]
    assert images.dtype == np.uint8 and images.ndim == 4 and images.shape[-1] == 3
    assert labels.ndim == 1 and np.issubdtype(labels.dtype, np.integer)
    assert len(images) == len(labels) == len(ids) == record["count"] > 0
    assert labels.min() >= 0 and labels.max() < len(classes)
    assert len(set(ids)) == len(ids) and not seen_ids.intersection(ids)
    seen_ids.update(ids)
    splits[split] = {"images": images, "labels": labels, "ids": ids}
    print(split, len(labels), "images")
print("클래스 순서:", classes)
```

`manifest.json`은 데이터의 클래스 순서, 분할, 전처리 설정을 설명하는 목록입니다. `classes`는 번호를 물체 이름으로 바꾸는 기준이 됩니다. 같은 이름이 중복되거나 클래스가 2개 미만이면 학습을 멈춥니다.

`identity`는 데이터의 정체를 설명하는 네 항목을 모은 사전입니다. 키 순서를 고정해 JSON으로 바꾸고 SHA-256을 계산하므로, 같은 기록을 같은 방식으로 비교할 수 있습니다. 다음 반복문은 `train`, `validation`, `test` 파일을 각각 확인합니다.

| 코드 묶음 | 확인하는 내용 |
|---|---|
| 파일 이름과 `sha256_file(file)` | manifest에 적힌 파일이 맞는가 |
| `np.load(..., allow_pickle=False)` | 숫자 배열을 읽는가 |
| `images.dtype`, `ndim`, `shape[-1]` | 사진이 정수 픽셀 배열이고 RGB 3채널인가 |
| `len(images) == len(labels) == len(ids)` | 사진마다 정답과 ID가 하나씩 있는가 |
| `labels.min()`, `labels.max()` | 정답 번호가 클래스 범위 안에 있는가 |
| `seen_ids.intersection(ids)` | 학습·검증·평가에 같은 ID가 섞이지 않았는가 |

`splits`에는 읽은 배열을 보관하고 `seen_ids`에는 지금까지 확인한 ID를 모읍니다. ID가 다르다는 검사만으로 비슷한 사진이나 복사된 이미지가 전혀 없다고 증명하는 것은 아닙니다.

**이 수업의 예상 형태:** 학습 사진은 `(500, 32, 32, 3)`, 정답은 `(500,)`입니다. 검증 100장, 최종 평가 200장을 따로 씁니다. 각 줄의 `train 500 images` 같은 출력과 클래스 순서를 확인합니다. 실습용 CIFAR-100 이미지의 성능을 실제 작업대 사진의 정확도로 해석하지 않습니다.

## 05. 전처리 도구와 사전학습 모델 불러오기

**원본 셀 번호:** `9`

**원본 코드 — 읽기 전용**

```python
processor = AutoImageProcessor.from_pretrained(
    MODEL_DIR, local_files_only=True, use_fast=False,
)
model = AutoModelForImageClassification.from_pretrained(
    MODEL_DIR, local_files_only=True, use_safetensors=False, weights_only=True,
    attn_implementation="eager",
).to(device)
print(type(model).__name__, "· 원래 분류 수:", model.config.num_labels)
print("전체 파라미터:", f"{sum(p.numel() for p in model.parameters()):,}")
```

두 `from_pretrained()`는 역할이 다릅니다. `processor`는 사진의 크기와 픽셀 값을 모델이 기대하는 입력으로 바꾸고, `model`은 그 입력을 받아 클래스별 점수를 계산합니다.

- `MODEL_DIR`를 주므로 이미 다운로드한 폴더를 읽습니다. `local_files_only=True`는 추가 다운로드를 하지 않도록 합니다.
- `use_fast=False`는 이 수업에서 사용하는 전처리 방식을 명시합니다.
- 이번 원본 모델에는 `pytorch_model.bin`이 있으므로 `use_safetensors=False`입니다. `weights_only=True`는 가중치를 읽는 방식을 제한합니다. 이 옵션 하나만으로 파일의 안전성이 증명되는 것은 아닙니다.
- `attn_implementation="eager"`는 수업에서 사용하는 attention 계산 방식을 고정합니다. `.to(device)`는 모델의 가중치를 GPU로 옮깁니다.

**실행 결과:** 모델 클래스 이름, 원래 분류 수 `1000`, 전체 파라미터 수가 나옵니다. `p.numel()`은 한 텐서 안에 들어 있는 숫자 개수이며 `sum(...)`은 모두 더합니다. 아직 5종류를 학습한 모델이 아닙니다. 뒤에서 마지막 분류층을 바꿉니다.

## 06. 사진을 모델 입력으로 바꾸고 16장씩 묶기

**원본 셀 번호:** `11`

**원본 코드 — 읽기 전용**

```python
TRAINING_ALLOWED = False
if (OUTPUT_DIR / "report.json").exists():
    raise FileExistsError("완료된 결과가 있습니다. results/gpu를 보관한 뒤 재실행하세요.")
import csv
import time
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 16
HEAD_EPOCHS, FINETUNE_EPOCHS = 3, 3
HEAD_LR, FINETUNE_LR = 1e-3, 1e-4
started = time.perf_counter()
loaders = {}
for name, split in splits.items():
    pil_images = [Image.fromarray(pixels).convert("RGB") for pixels in split["images"]]
    pixels = processor(images=pil_images, return_tensors="pt")["pixel_values"]
    dataset = TensorDataset(pixels, torch.tensor(split["labels"], dtype=torch.long))
    loaders[name] = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=(name == "train"),
        generator=torch.Generator().manual_seed(SEED), num_workers=0,
    )
print("입력 크기:", tuple(loaders["train"].dataset.tensors[0].shape))
TRAINING_ALLOWED = True
```

이 셀은 이전 결과를 보호한 뒤 학습 데이터를 준비합니다. `report.json`이 이미 있으면 멈추므로, 실제 재실행 전에는 완료한 결과를 다른 위치에 보관합니다. `TRAINING_ALLOWED`는 모든 준비가 끝난 마지막 줄에서만 `True`가 됩니다.

`BATCH_SIZE = 16`은 한 번의 업데이트에 최대 16장을 넣는다는 뜻입니다. epoch는 학습 사진 전체를 한 차례 읽는 단위이고, 두 단계 모두 3회 반복합니다. `1e-3`은 `0.001`, `1e-4`는 `0.0001`입니다. 학습률은 가중치를 얼마나 움직일지 조절합니다. `started`는 실행 시간을 재기 위한 시작 시각입니다.

데이터가 지나가는 순서를 따라 읽어 보세요.

1. `Image.fromarray(...).convert("RGB")`: 숫자 배열을 PIL 이미지로 바꾸고 색 채널을 맞춥니다.
2. `processor(..., return_tensors="pt")["pixel_values"]`: 이미지 크기와 픽셀 값을 맞춰 PyTorch 텐서로 만듭니다. 학습 입력은 `(500, 3, 224, 224)`입니다. 순서는 사진 수, 색 채널, 높이, 너비입니다.
3. `TensorDataset(pixels, labels)`: 사진과 정답 번호가 항상 함께 나오도록 짝지어 둡니다. `torch.long`은 CrossEntropyLoss에 필요한 정수 정답 형식입니다.
4. `DataLoader`: 짝지은 데이터를 16장씩 꺼내 줍니다. 학습 데이터만 섞고 검증·평가는 순서를 유지합니다. `num_workers=0`은 별도의 로딩 작업 프로세스를 만들지 않는 설정입니다.

**실행 결과:** `입력 크기: (500, 3, 224, 224)`가 나옵니다. 500은 16으로 나누어떨어지지 않아 마지막 배치는 4장입니다. 여기서는 나머지를 버리거나 가짜 사진을 채우지 않습니다. 모델은 앞에서 GPU로 옮겼습니다. 배치는 뒤의 학습·평가 반복문에서 GPU로 옮깁니다.

### 작은 CPU 연습 · 500장을 16장씩 나누기

아래 숫자는 원본의 배치 구성을 계산하는 연습입니다. 사진을 읽거나 GPU를 사용하지 않습니다. `batch_size`를 32로 바꾸면 마지막 배치는 몇 장일지 먼저 예상해 보세요.

**처음 설정의 예상 출력:** 배치 32개, 마지막 배치 4장, 합계 500장.

In [1]:
total_images = 500
batch_size = 16
batch_sizes = [min(batch_size, total_images - start)
               for start in range(0, total_images, batch_size)]
print("배치 수:", len(batch_sizes))
print("마지막 배치:", batch_sizes[-1])
print("총 사진 수:", sum(batch_sizes))

배치 수: 32
마지막 배치: 4
총 사진 수: 500


## 07. 5종류를 분류할 새 헤드 만들기

**원본 셀 번호:** `13`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
for parameter in model.parameters():
    parameter.requires_grad = False
model.classifier = nn.Linear(model.config.hidden_size, len(classes)).to(device)
nn.init.normal_(model.classifier.weight, std=model.config.initializer_range)
nn.init.zeros_(model.classifier.bias)
model.num_labels = len(classes)
model.config.num_labels = len(classes)
model.config.id2label = dict(enumerate(classes))
model.config.label2id = {label: index for index, label in enumerate(classes)}
criterion = nn.CrossEntropyLoss()

def parameter_sha256(named_parameters):
    digest = hashlib.sha256()
    for name, parameter in named_parameters:
        digest.update(name.encode())
        digest.update(parameter.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

backbone_before = parameter_sha256(model.vit.named_parameters())
history = []
print("헤드 학습 파라미터:", sum(p.numel() for p in model.parameters() if p.requires_grad))
```

모델의 앞부분은 사진의 특징을 추출하고, 마지막 `classifier`는 그 특징으로 물체 종류별 점수를 만듭니다. 먼저 기존 파라미터의 `requires_grad`를 모두 `False`로 바꿔 학습 대상에서 제외합니다. 그 다음 새로 만든 `nn.Linear`는 기본적으로 학습 가능한 상태라 분류 헤드만 학습할 수 있습니다.

`nn.Linear(model.config.hidden_size, len(classes))`는 192개의 특징을 5개 점수로 바꿉니다. 가중치는 `192 × 5`개, bias는 5개라 총 `965`개입니다. `normal_`은 작은 난수로 가중치를 초기화하고 `zeros_`는 bias를 0으로 만듭니다. 함수 이름 끝의 `_`는 대체로 원래 값을 직접 바꾸는 연산을 뜻합니다.

`num_labels`, `id2label`, `label2id`도 함께 바꿔 모델의 출력 수와 클래스 이름을 맞춥니다. 예를 들어 `id2label[0]`은 0번 물체 이름입니다. 이 설정은 모델을 저장했다가 다시 불러올 때도 필요합니다.

`CrossEntropyLoss()`는 정답 번호와 **softmax를 적용하기 전의 점수인 logits**를 받아 손실을 계산합니다. 학생 함수에서 logits에 미리 softmax를 적용할 필요가 없습니다.

`parameter_sha256()`은 이름과 가중치 숫자들을 순서대로 묶어 비교용 SHA-256을 계산합니다. `detach()`는 자동 미분 연결을 떼고, `cpu()`는 CPU로 옮기고, `numpy().tobytes()`는 숫자들을 바이트로 읽습니다. `backbone_before`는 헤드 외의 부분이 실제로 고정되었는지 나중에 비교할 기록입니다. `history`에는 각 epoch의 성능을 모읍니다.

**실행 결과:** `헤드 학습 파라미터: 965`가 나옵니다. “고정”은 이번 학습에서 값을 업데이트하지 않겠다는 뜻이며 사진 계산에서 제외한다는 뜻은 아닙니다.

### 작은 CPU 연습 · 헤드의 숫자가 왜 965개일까?

모든 입력 특징이 각 출력 점수와 연결되므로 가중치는 입력 수 × 출력 수입니다. bias는 출력마다 하나입니다. 클래스 수를 6으로 바꾸면 **1,158개**가 됩니다. 실제 모델은 수정하지 않습니다.

**기본 예상 출력:** 가중치 960개, bias 5개, 합계 965개.

In [2]:
input_features = 192
class_count = 5
weight_count = input_features * class_count
bias_count = class_count
print("가중치:", weight_count)
print("bias:", bias_count)
print("합계:", weight_count + bias_count)

가중치: 960
bias: 5
합계: 965


## 08. 학습하지 않고 손실과 정확도 계산하기

**원본 셀 번호:** `15`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
def evaluate(data_loader):
    model.eval()
    loss_sum, all_labels, all_predictions = 0.0, [], []
    with torch.inference_mode():
        for pixels, labels in data_loader:
            pixels, labels = pixels.to(device), labels.to(device)
            logits = model(pixel_values=pixels).logits
            loss_sum += criterion(logits, labels).item() * len(labels)
            all_labels.extend(labels.cpu().tolist())
            all_predictions.extend(logits.argmax(dim=-1).cpu().tolist())
    labels = np.asarray(all_labels)
    predictions = np.asarray(all_predictions)
    matrix = np.zeros((len(classes), len(classes)), dtype=int)
    np.add.at(matrix, (labels, predictions), 1)
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    f1 = np.divide(2 * matrix.diagonal(), denominator,
                   out=np.zeros(len(classes)), where=denominator > 0)
    return {
        "loss": loss_sum / len(labels), "accuracy": float(np.mean(labels == predictions)),
        "macro_f1": float(f1.mean()), "count": len(labels),
        "confusion_matrix": matrix.tolist(), "predictions": predictions.tolist(),
    }
```

`def evaluate(data_loader)`는 평가할 때 반복해서 부를 기능을 정의합니다. 함수를 정의하는 것만으로 사진 평가가 시작되지는 않습니다. 뒤에서 `evaluate(loaders["validation"])`처럼 호출합니다.

**앞부분: 예측을 모읍니다.** `model.eval()`은 모델을 평가 모드로 바꿉니다. `torch.inference_mode()`는 이 구간에서 학습용 기울기를 기록하지 않도록 합니다. 두 기능은 다릅니다. `eval()`만으로 `requires_grad`가 꺼지거나 학습 대상이 고정되는 것은 아닙니다.

반복문에서 사진과 정답을 GPU로 옮기고 `(이번 배치의 사진 수, 5)` logits를 얻습니다. `argmax(dim=-1)`는 각 사진의 가장 큰 점수 위치를 정답 예측 번호로 고릅니다. `extend`는 목록에 배치별 값을 이어 붙입니다. 평균 손실에 사진 수를 곱해 합산하는 것은 마지막 배치가 4장일 수 있기 때문입니다.

**뒷부분: 결과를 정리합니다.** `matrix`는 5×5 혼동행렬입니다. `np.add.at(matrix, (labels, predictions), 1)`은 각 사진의 **행=실제 정답, 열=모델 예측** 위치에 1을 더합니다. 대각선은 맞힌 사진 수입니다.

클래스별 F1은 `2 × 맞힌 수 / (그 클래스로 예측한 수 + 실제 그 클래스 수)`로 계산합니다. `axis=0`은 열 합계, `axis=1`은 행 합계입니다. 분모가 0인 클래스는 `where=denominator > 0`과 0으로 채운 `out`을 써서 0으로 처리합니다. `macro_f1`은 클래스별 F1의 단순 평균입니다.

**반환값:** 손실, 정확도, Macro F1, 사진 수, 혼동행렬, 각 사진의 예측 번호가 담긴 사전입니다. 정확도와 F1은 비슷한 역할을 하지만 같은 수식이 아닙니다. `loss_sum / len(labels)`처럼 마지막에 전체 사진 수로 나눠 크기가 다른 배치도 공평하게 반영합니다.

### 작은 CPU 연습 · 마지막 4장도 올바르게 평균내기

설명을 위해 **16장짜리 한 배치와 4장짜리 한 배치만** 있다고 가정합니다. 손실 0.4, 0.8은 임의로 만든 값입니다. 두 평균을 단순히 더해 나누면 사진 수가 다른데도 같은 비중이 됩니다.

**예상 출력:** 사진 수를 반영한 평균은 0.480, 배치 평균의 단순 평균은 0.600입니다.

In [3]:
batch_results = [(16, 0.4), (4, 0.8)]
total_loss = sum(count * loss for count, loss in batch_results)
total_count = sum(count for count, loss in batch_results)
weighted_mean = total_loss / total_count
simple_mean = sum(loss for count, loss in batch_results) / len(batch_results)
print(f"사진 수 반영: {weighted_mean:.3f}")
print(f"배치별 단순 평균: {simple_mean:.3f}")

사진 수 반영: 0.480
배치별 단순 평균: 0.600


## 09. 정답을 확인하기 쉬운 작은 연습 모델 만들기

**원본 셀 번호:** `17`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
from types import SimpleNamespace
EXERCISE_CHECKS["02-step"] = False

class PracticeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = nn.Linear(2, 2)

    def forward(self, pixel_values):
        return SimpleNamespace(logits=self.classifier(pixel_values))

def make_practice_model():
    # 연습 모델 초기화가 본학습의 난수 순서를 바꾸지 않도록 합니다.
    with torch.random.fork_rng(devices=[]):
        practice = PracticeClassifier().to(device)
    with torch.no_grad():
        practice.classifier.weight.zero_()
        practice.classifier.bias.zero_()
    return practice.eval()

practice_pixels = torch.tensor([[1.0, 0.0], [0.0, 1.0]], device=device)
practice_labels = torch.tensor([0, 1], dtype=torch.long, device=device)
print("연습 입력:", practice_pixels.tolist(), "· 정답:", practice_labels.tolist())
print("연습 optimizer: SGD, 학습률 0.1 / 실제 본학습: 기존 Adam 설정 유지")
```

실제 사진 모델은 수백만 개 숫자를 쓰므로 학생 함수가 잘못되었을 때 원인을 찾기 어렵습니다. 이 셀에서는 2개 숫자를 받아 2개 점수를 내는 작은 모델을 준비합니다. 뒤에서 작성할 함수를 검사하는 용도이며 본학습용 모델을 바꾸지 않습니다.

`PracticeClassifier(nn.Module)`는 PyTorch 모델의 기본 기능을 물려받습니다. `__init__`는 모델을 만들 때 실행되는 준비 단계이고, `forward`는 입력을 넣었을 때 계산하는 방법입니다. `SimpleNamespace(logits=...)`를 반환해 실제 모델과 같은 `.logits` 접근 방식을 제공합니다.

`make_practice_model()`은 연습 모델을 새로 만들고 가중치·bias를 모두 0으로 맞춥니다. `fork_rng(devices=[])`는 CPU 난수 상태를 보존해 연습 모델 생성이 본학습의 난수 순서를 바꾸지 않도록 합니다. `.eval()`로 시작하므로 학생 함수가 학습 모드로 바꾸는지도 검사할 수 있습니다.

입력 `[[1, 0], [0, 1]]`은 사진이 아니라 원리를 확인하기 위한 숫자 두 쌍입니다. 정답 `[0, 1]`은 첫 입력의 정답은 0번, 두 번째는 1번이라는 뜻입니다. 둘 다 GPU에 놓습니다.

**실행 결과:** 연습 입력과 정답, 연습은 `SGD`·학습률 `0.1`이고 실제 학습은 `Adam`이라는 안내가 출력됩니다. 두 optimizer는 가중치 업데이트 방법이 다르므로 연습에 쓴 SGD로 본학습 설정을 덮어쓰지 않습니다.

## 10. 빈칸 실습 1 — 한 배치에서 가중치를 한 번 바꾸기

**원본 셀 번호:** `19`

**원본 코드 — 학생이 채울 빈 실습 셀**

```python

```

원본의 이 셀은 의도적으로 비어 있습니다. 아래는 완성 코드를 제공하는 대신 학생 함수가 해야 할 일을 풀어 쓴 계약입니다. 원본의 **실습 1** 문제와 AI 프롬프트를 읽고 이 셀에 `train_one_batch(model, optimizer, criterion, pixels, labels)`를 작성합니다.

| 들어오는 값 | 뜻 |
|---|---|
| `model` | 이미 준비된 모델 |
| `optimizer` | 이미 준비된 업데이트 도구와 학습률 |
| `criterion` | logits와 정답을 비교할 손실 함수 |
| `pixels`, `labels` | 같은 장치에 있는 이번 배치의 입력·정답 |

한 번의 학습은 “학습 모드 → 이전 기울기 지우기 → 점수 계산 → 손실 계산 → 기울기 계산 → 업데이트” 순서로 읽습니다. `backward`는 바꿀 방향을 계산하고 `step`은 optimizer가 실제 값을 바꿉니다. `zero_grad`를 빠뜨리면 이전 배치의 기울기가 쌓입니다.

돌려줄 것은 `(손실 float, 맞힌 수 int, 입력 수 int)`입니다. 손실과 정답 개수는 **이번 업데이트 전**의 logits를 기준으로 계산합니다. 새 모델이나 optimizer를 만들거나 예상 숫자를 답으로 고정하면 조건을 충족하지 못합니다.

**시도할 질문:** “점수만 계산하고 업데이트를 안 하면 다음 호출의 손실은 달라질까?”, “기울기를 지우는 작업을 한 번만 하면 왜 안 될까?”를 AI에게 설명하게 해 보세요. 다음 원본 확인 셀이 이 조건들을 작은 입력으로 검사합니다.

### 작은 CPU 연습 · 기울기와 업데이트는 다른 단계

가중치 숫자 하나 `w`를 정답 1에 가깝게 바꾸는 장난감 예제입니다. 손실은 `(w - 1)²`, 그 기울기는 `2 × (w - 1)`입니다. **실제 노트북의 CrossEntropyLoss·Adam이나 학생 함수 답안이 아닙니다.** 계산 방향과 값 변경을 구분하는 데만 사용합니다.

**예상 출력:** 업데이트 전 0.0, 기울기 -2.0, 업데이트 후 0.2, 손실 1.000 → 0.640입니다.

In [4]:
weight = 0.0
target = 1.0
learning_rate = 0.1
loss_before = (weight - target) ** 2
gradient = 2 * (weight - target)
new_weight = weight - learning_rate * gradient
loss_after = (new_weight - target) ** 2
print(f"업데이트 전: {weight:.1f}")
print(f"기울기: {gradient:.1f}")
print(f"업데이트 후: {new_weight:.1f}")
print(f"손실: {loss_before:.3f} → {loss_after:.3f}")

업데이트 전: 0.0
기울기: -2.0
업데이트 후: 0.2
손실: 1.000 → 0.640


## 11. 학생 함수가 실제로 업데이트했는지 확인하기

**원본 셀 번호:** `20`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
EXERCISE_CHECKS["02-step"] = False
if not callable(globals().get("train_one_batch")):
    raise RuntimeError("실습 1의 빈 셀에 train_one_batch 함수를 만들고 실행하세요.")
practice = make_practice_model()
practice_optimizer = torch.optim.SGD(practice.parameters(), lr=0.1)
first = train_one_batch(practice, practice_optimizer, criterion, practice_pixels, practice_labels)
second = train_one_batch(practice, practice_optimizer, criterion, practice_pixels, practice_labels)
for result in (first, second):
    assert isinstance(result, tuple) and len(result) == 3, "튜플 (손실, 맞힌 수, 사진 수)을 반환하세요."
    assert type(result[0]) is float and all(type(x) is int for x in result[1:])
assert practice.training, "함수 안에서 학습 모드로 바꾸세요."
assert abs(first[0] - 0.69314718) < 1e-5 and first[1:] == (1, 2)
assert abs(second[0] - 0.66845965) < 1e-5 and second[1:] == (2, 2)
expected_weight = torch.tensor([[0.04937513, -0.04937513], [-0.04937513, 0.04937513]], device=device)
torch.testing.assert_close(practice.classifier.weight, expected_weight, rtol=1e-5, atol=1e-6)
torch.testing.assert_close(practice.classifier.bias, torch.zeros(2, device=device), rtol=0, atol=1e-6)
EXERCISE_CHECKS["02-step"] = True
print(f"1회: loss={first[0]:.4f}, correct={first[1]}, count={first[2]}")
print(f"2회: loss={second[0]:.4f}, correct={second[1]}, count={second[2]}")
print("실습 1 확인 통과: 두 번의 업데이트와 기울기 초기화 결과가 맞습니다.")
```

`callable(...)`은 학생이 만든 이름이 호출 가능한 함수인지 확인합니다. 없으면 GPU 본학습에 들어가기 전에 멈춥니다. 검사 시작 시 `EXERCISE_CHECKS["02-step"]`를 `False`로 돌리고 모든 검사가 끝난 뒤에만 `True`로 바꿉니다.

`make_practice_model()`로 조건을 맞춘 모델과 SGD를 만들고 **같은 모델**에 학생 함수를 두 번 호출합니다. 두 번째 호출은 첫 번째에서 바뀐 가중치를 이어서 사용합니다. 반환 자료형은 튜플인지, 손실은 Python `float`인지, 두 개수는 `int`인지 확인합니다.

0으로 시작한 logits는 두 클래스의 점수가 같습니다. 첫 예측은 `argmax`의 첫 위치 0을 선택하므로 정답 `[0, 1]` 중 1개만 맞힙니다. 첫 손실은 약 `0.6931`입니다. 두 번의 지정된 업데이트가 맞으면 두 번째 손실은 약 `0.6685`, 맞힌 수는 2개가 됩니다. 이 숫자는 **이 작은 고정 입력의 검사 기준**이며 실제 사진의 성능 예측이 아닙니다.

`assert_close`는 연습 모델의 실제 가중치까지 기대값과 비교합니다. `rtol`과 `atol`은 부동소수점 계산의 작은 차이를 허용하는 상대·절대 오차 범위입니다. 출력 숫자만 흉내 낸 함수가 통과하지 않도록 값도 확인하는 셀입니다.

**오류를 읽는 순서:** 반환 형식 → 학습 모드 → 손실·개수 → 업데이트된 가중치 순서로 봅니다. 두 번째 결과만 틀리면 기울기 초기화 누락이나 업데이트 횟수부터 점검해 보세요.

## 12. 학습률을 하나씩 바꿔 비교하기

**원본 셀 번호:** `22`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not EXERCISE_CHECKS.get("02-step", False):
    raise RuntimeError("실습 1의 확인 셀을 먼저 통과하세요.")
for practice_lr in (0.01, 0.1):
    trial_model = make_practice_model()
    trial_optimizer = torch.optim.SGD(trial_model.parameters(), lr=practice_lr)
    losses = []
    for _ in range(2):
        result = train_one_batch(trial_model, trial_optimizer, criterion, practice_pixels, practice_labels)
        losses.append(result[0])
    print(f"연습 학습률 {practice_lr}: {losses[0]:.4f} → {losses[1]:.4f}")
```

이 셀은 실습 1의 검사를 통과한 뒤에만 실행됩니다. `for practice_lr in (0.01, 0.1)`은 두 학습률을 차례로 비교합니다. 각 비교마다 새 연습 모델을 만들므로 둘 다 같은 출발점에서 시작합니다.

안쪽 `range(2)`는 같은 모델에서 두 번 학습합니다. `_`는 반복 번호를 계산에 쓰지 않겠다는 관례적인 이름입니다. `losses.append(result[0])`는 반환 튜플의 첫 항목인 손실만 기록합니다.

**읽을 결과:** 두 경우의 첫 손실은 약 `0.6931`로 같고, 두 번째 손실에서 차이가 납니다. 이 작은 예제에서는 `0.1`이 더 빨리 손실을 낮춥니다. 큰 학습률이 모든 데이터와 모델에서 좋다는 결론은 낼 수 없습니다. 실제 헤드 학습은 `Adam`·`0.001`을 그대로 사용합니다.

**다시 생각하기:** 두 모델을 새로 만들지 않고 첫 모델에서 계속 이어 했다면 학습률만의 차이를 비교할 수 있을까요? 출발 가중치까지 달라지므로 공정한 비교가 어려워집니다.

## 13. 헤드를 학습하고 검증 점수가 가장 좋은 시점 고르기

**원본 셀 번호:** `24`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not EXERCISE_CHECKS.get("02-step", False):
    raise RuntimeError("실습 1의 확인 셀을 먼저 통과하세요.")
HEAD_STAGE_COMPLETE = False
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=HEAD_LR)
best_key, best_head, best_head_epoch = (-1.0, float("-inf")), None, None
for epoch in range(1, HEAD_EPOCHS + 1):
    model.train()
    train_loss, train_correct, train_count = 0.0, 0, 0
    for pixels, labels in loaders["train"]:
        pixels, labels = pixels.to(device), labels.to(device)
        loss_value, correct_count, batch_count = train_one_batch(model, optimizer, criterion, pixels, labels)
        train_loss += loss_value * batch_count
        train_correct += correct_count
        train_count += batch_count
    validation = evaluate(loaders["validation"])
    history.append({"stage": "head", "epoch": epoch, "train_loss": train_loss / train_count,
                    "train_accuracy": train_correct / train_count,
                    "validation_loss": validation["loss"], "validation_accuracy": validation["accuracy"]})
    key = (validation["accuracy"], -validation["loss"])
    if key > best_key:
        best_key, best_head_epoch = key, epoch
        best_head = {name: value.detach().cpu().clone()
                     for name, value in model.classifier.state_dict().items()}
    print(f"head {epoch}/{HEAD_EPOCHS} · validation {validation['accuracy']:.1%}")
model.classifier.load_state_dict(best_head)
assert parameter_sha256(model.vit.named_parameters()) == backbone_before
head_validation = evaluate(loaders["validation"])
head_test = evaluate(loaders["test"])
print(f"선택 epoch {best_head_epoch} · head test {head_test['accuracy']:.1%}")
HEAD_STAGE_COMPLETE = True
```

이 셀부터는 연습용 숫자가 아닌 실제 사진으로 학습합니다. `Adam(model.classifier.parameters(), lr=HEAD_LR)`은 헤드의 가중치만 업데이트합니다. `HEAD_STAGE_COMPLETE`는 전체 학습·평가를 성공한 마지막에만 참이 됩니다.

**epoch 안쪽:** 16장씩 GPU로 옮겨 학생이 만든 `train_one_batch`를 호출합니다. 배치의 평균 손실에 배치 사진 수를 곱해 합산한 뒤 전체 사진 수로 나눕니다. 맞힌 수와 사진 수도 각각 더합니다. `history.append(...)`에는 이번 epoch의 학습·검증 성능을 기록합니다.

**가장 좋은 시점 선택:** `key = (검증 정확도, -검증 손실)`는 두 값을 순서대로 비교하는 튜플입니다. 먼저 정확도가 높은 것을 고르고, 같을 때는 음수로 바꾼 손실이 큰 것, 즉 원래 손실이 작은 것을 고릅니다. `(-1.0, -inf)`는 첫 정상 결과가 후보가 되도록 만든 시작값입니다. 두 값이 모두 같으면 `>`가 성립하지 않아 먼저 골랐던 epoch를 유지합니다.

`state_dict()`는 헤드 가중치의 이름과 값입니다. `detach().cpu().clone()`은 학습 그래프에서 떼어 CPU의 별도 저장 공간에 복사합니다. 그냥 같은 객체를 기억하면 다음 epoch에서 값이 바뀌어 “그때의 최고 모델”을 잃을 수 있습니다.

**반복이 끝나면:** `load_state_dict(best_head)`로 선택한 헤드를 복원하고, 앞부분의 SHA-256이 처음과 같은지 확인합니다. 그 후 검증·최종 평가를 실행합니다. 테스트 정확도를 보고 epoch를 고르는 코드는 없습니다.

**실행 결과:** epoch별 검증 정확도와 선택된 epoch의 테스트 정확도가 출력됩니다. 마지막 epoch가 반드시 선택되는 것도, 정확도가 계속 오르는 것도 아닙니다.

### 작은 CPU 연습 · 정확도가 같으면 손실을 비교하기

다음은 설명용 가상 검증 기록입니다. 2·3 epoch의 정확도가 같지만 2 epoch의 손실이 더 낮습니다. 마지막 epoch를 무조건 고르지 않는 이유를 확인합니다.

**예상 출력:** 선택 epoch 2. 이 숫자는 실제 학습에서 선택될 epoch의 예측이 아닙니다.

In [5]:
records = [
    {"epoch": 1, "accuracy": 0.70, "loss": 0.65},
    {"epoch": 2, "accuracy": 0.80, "loss": 0.45},
    {"epoch": 3, "accuracy": 0.80, "loss": 0.50},
]
best = max(records, key=lambda row: (row["accuracy"], -row["loss"]))
print("선택 epoch:", best["epoch"])
print("선택 기준:", (best["accuracy"], -best["loss"]))

선택 epoch: 2
선택 기준: (0.8, -0.45)


### 작은 CPU 연습 · 최고 모델은 별도로 복사해야 한다

숫자 목록으로 복사의 필요성을 봅니다. `same_object`는 같은 목록을 가리키고 `snapshot`은 별도 목록입니다. 실제 코드의 텐서에서는 `.clone()`이 값의 복사 역할을 합니다. `.cpu()`만으로 항상 별도 복사본이 생긴다고 가정하지 않습니다.

**예상 출력:** 같은 객체는 `[9.0, 2.0]`, 복사본은 `[1.0, 2.0]`을 유지합니다.

In [6]:
weights = [1.0, 2.0]
same_object = weights
snapshot = weights.copy()
weights[0] = 9.0
print("같은 객체:", same_object)
print("복사한 기록:", snapshot)

같은 객체: [9.0, 2.0]
복사한 기록: [1.0, 2.0]


## 14. 추가 학습을 시작해도 되는지 확인하기

**원본 셀 번호:** `26`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
EXERCISE_CHECKS["02-scope"] = False
FINETUNE_SETUP_COMPLETE = False
if not HEAD_STAGE_COMPLETE:
    raise RuntimeError("분류 헤드 학습과 평가를 먼저 완료하세요.")
print("완료한 분류 헤드의 검증 정확도:", f"{head_validation['accuracy']:.1%}")
print("Transformer 블록 수:", len(model.vit.encoder.layer))
print("분류할 종류:", model.config.num_labels)
```

이 셀은 실습 2에 필요한 선행 상태를 확인합니다. 데이터 준비를 마쳤고 `HEAD_STAGE_COMPLETE`가 참이어야 진행합니다. 헤드를 학습하지 않은 상태에서 곧바로 마지막 블록을 바꾸는 실수를 막습니다.

`EXERCISE_CHECKS["02-scope"] = False`와 `FINETUNE_SETUP_COMPLETE = False`는 범위 선택과 파인튜닝 준비를 새로 확인하겠다는 표시입니다. 가중치를 초기화하는 코드는 없습니다. 앞에서 선택한 가장 좋은 헤드가 그대로 남아 있습니다.

**실행 결과:** 완료한 헤드의 검증 정확도, Transformer 블록 수 `12`, 분류할 종류 `5`를 보여 줍니다. Python의 번호는 0부터 시작하므로 마지막 블록의 번호는 11이고 `layer[-1]`로도 접근합니다. 실제 검증 정확도는 실행 결과를 읽습니다.

## 15. 빈칸 실습 2 — 어느 부분을 더 학습할지 고르기

**원본 셀 번호:** `28`

**원본 코드 — 학생이 채울 빈 실습 셀**

```python

```

이 셀도 원본에서 의도적으로 비어 있습니다. 원본 **실습 2**의 조건과 프롬프트를 보고 `select_finetune_parameters(model, scope)`를 작성합니다. 이미 학습한 가중치를 유지한 채 이번에 업데이트할 범위만 고르는 문제입니다.

| `scope` 값 | 학습 가능한 부분 | 이 모델에서 숫자 개수 |
|---|---|---:|
| `"head"` | classifier | 965 |
| `"last_block"` | 마지막 Transformer 블록 + 최종 layernorm + classifier | 446,213 |

호출할 때마다 전체 `requires_grad`를 먼저 거짓으로 돌리고 선택한 부분만 참으로 바꿔야 합니다. 그렇지 않으면 `last_block` 다음 `head`를 골라도 이전 선택이 남을 수 있습니다. `layer[-1]`은 마지막 블록이며, 모델 크기가 달라져도 마지막을 가리키게 하려는 표현입니다.

반환값은 `{파라미터 이름: 실제 parameter 객체}` 사전입니다. 이름은 `named_parameters()`에서 읽고 원래 객체를 그대로 전달해야 합니다. 숫자 개수만 반환하거나 복사한 새 텐서를 주면 optimizer가 실제 모델의 원하는 가중치를 업데이트하지 못합니다.

**조건을 시험하기:** 지원하지 않는 `scope`는 `ValueError`로 알리고, 범위 선택 중 가중치 값 자체는 바꾸지 않습니다. `head → last_block → head`로 호출했을 때 마지막 개수가 왜 다시 965여야 하는지 설명해 보세요. 이 해설에는 함수의 완성 답안을 넣지 않았습니다.

## 16. 범위를 여러 번 바꿔도 선택이 정확한지 검사하기

**원본 셀 번호:** `29`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
EXERCISE_CHECKS["02-scope"] = False
if not HEAD_STAGE_COMPLETE:
    raise RuntimeError("분류 헤드 학습을 먼저 완료하세요.")
if not callable(globals().get("select_finetune_parameters")):
    raise RuntimeError("실습 2의 빈 셀에 select_finetune_parameters 함수를 작성하세요.")
unchanged_weights = parameter_sha256(model.named_parameters())
last_prefix = f"vit.encoder.layer.{len(model.vit.encoder.layer) - 1}."
parameter_lookup = dict(model.named_parameters())
for scope in ("head", "last_block", "head", "last_block"):
    chosen = select_finetune_parameters(model, scope)
    prefixes = ("classifier.",) if scope == "head" else (last_prefix, "vit.layernorm.", "classifier.")
    expected = {name for name in parameter_lookup if name.startswith(prefixes)}
    assert isinstance(chosen, dict) and set(chosen) == expected, "선택한 가중치의 이름을 확인하세요."
    assert all(chosen[name] is parameter_lookup[name] for name in expected), "원래 parameter 객체를 반환하세요."
    assert {name for name, p in model.named_parameters() if p.requires_grad} == expected
    assert parameter_sha256(model.named_parameters()) == unchanged_weights, "가중치 값은 바꾸지 마세요."
    print(f"범위 {scope}: {sum(p.numel() for p in chosen.values()):,}개 파라미터")
try:
    select_finetune_parameters(model, "unknown")
except ValueError:
    pass
else:
    raise AssertionError("지원하지 않는 범위에는 ValueError가 필요합니다.")
# 위 오류 입력까지 확인한 뒤 본학습 범위를 다시 적용합니다.
chosen = select_finetune_parameters(model, "last_block")
assert set(chosen) == expected
assert {n for n, p in model.named_parameters() if p.requires_grad} == expected
assert all(chosen[name] is parameter_lookup[name] for name in expected)
assert parameter_sha256(model.named_parameters()) == unchanged_weights
EXERCISE_CHECKS["02-scope"] = True
print("실습 2 확인 통과: 범위를 다시 바꿔도 고정 상태와 원래 가중치를 유지했습니다.")
```

이 셀은 학생 함수가 “처음 한 번”만 맞는 것이 아니라 반복해서 바꿔도 맞는지 확인합니다. `unchanged_weights`에 현재 전체 가중치의 SHA-256을 저장하고, `parameter_lookup`에는 이름과 실제 객체를 모아 둡니다.

`last_prefix`는 마지막 블록의 이름 앞부분입니다. `startswith(prefixes)`는 파라미터 이름이 허용한 부분으로 시작하는지 확인합니다. `expected`는 정답으로 기대하는 **이름들의 집합**이지 가중치 값 목록이 아닙니다.

반복문은 `head → last_block → head → last_block`을 시험하며 네 가지를 검사합니다.

1. 반환한 것이 사전이고 이름이 기대한 범위와 같은가?
2. `is`로 비교했을 때 원래 모델에 속한 같은 parameter 객체인가?
3. 실제 `requires_grad=True`인 이름도 그 범위와 일치하는가?
4. 가중치 값은 범위를 고르기 전과 같은가?

`try/except`는 일부러 잘못된 `"unknown"` 값을 넣어 `ValueError`가 나는지 확인합니다. 그 오류 처리까지 검사한 후 본학습 범위를 `last_block`으로 다시 맞춥니다.

**예상 출력:** `965 → 446,213 → 965 → 446,213`입니다. 모두 통과해야 확인 표시를 `True`로 바꿉니다. 값이 많아졌다는 사실은 “더 많은 숫자를 수정한다”는 뜻이며 성능이 더 좋다는 뜻은 아닙니다.

## 17. 마지막 블록 학습을 준비하고 비교 기준 남기기

**원본 셀 번호:** `31`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
FINETUNE_SETUP_COMPLETE = False
if not HEAD_STAGE_COMPLETE or not all(EXERCISE_CHECKS.values()):
    raise RuntimeError("두 실습 확인과 분류 헤드 학습을 먼저 완료하세요.")
last_block_index = len(model.vit.encoder.layer) - 1
last_block_prefix = f"vit.encoder.layer.{last_block_index}."
trainable = select_finetune_parameters(model, "last_block")
frozen_before = parameter_sha256((n, p) for n, p in model.named_parameters() if not p.requires_grad)
tail_before = parameter_sha256(model.vit.encoder.layer[-1].named_parameters())
trainable_count = sum(parameter.numel() for parameter in trainable.values())
assert all(name.startswith((last_block_prefix, "vit.layernorm.", "classifier.")) for name in trainable)
print("파인튜닝 파라미터:", f"{trainable_count:,}")
optimizer = torch.optim.Adam(trainable.values(), lr=FINETUNE_LR)
best_key, best_tail, best_finetune_epoch = (-1.0, float("-inf")), None, None
FINETUNE_SETUP_COMPLETE = True
```

이 셀은 헤드 학습과 두 실습 검사를 모두 마친 뒤 실행됩니다. `all(EXERCISE_CHECKS.values())`는 모든 확인 항목이 참인지 검사합니다. `FINETUNE_SETUP_COMPLETE`는 준비가 성공했을 때만 참이 됩니다.

`select_finetune_parameters(model, "last_block")`가 돌려준 사전은 이제 실제 Adam에 전달할 학습 대상입니다. `sum(parameter.numel() ...)`은 수정할 숫자 수를 계산하며 현재 설정에서 `446,213`개입니다. 앞의 11개 Transformer 블록을 포함한 나머지 부분은 고정합니다.

`frozen_before`는 고정한 가중치가 끝까지 같은지 검사할 기준이고, `tail_before`는 마지막 블록이 실제로 바뀌었는지 검사할 기준입니다. `assert all(...)`은 선택된 이름이 허용한 접두사에 속하는지 추가로 확인합니다.

`Adam(trainable.values(), lr=FINETUNE_LR)`는 이 단계용 optimizer를 만듭니다. 헤드 학습에서 고른 가중치부터 이어서 시작하지만 optimizer는 새로 생성합니다. 학습률은 `0.0001`로 설정합니다. `best_key`, `best_tail`, `best_finetune_epoch`는 이 단계의 최고 검증 결과를 따로 기록할 변수입니다.

**실행 결과:** 파인튜닝 파라미터 수가 출력됩니다. 아직 이 셀에서는 학습 반복문을 실행하지 않습니다.

## 18. 선택한 부분을 학습하고 고정 여부 확인하기

**원본 셀 번호:** `32`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
FINETUNE_STAGE_COMPLETE = False
if not FINETUNE_SETUP_COMPLETE or not all(EXERCISE_CHECKS.values()):
    raise RuntimeError("두 실습 확인과 파인튜닝 준비를 먼저 완료하세요.")
for epoch in range(1, FINETUNE_EPOCHS + 1):
    model.train()
    train_loss, train_correct, train_count = 0.0, 0, 0
    for pixels, labels in loaders["train"]:
        pixels, labels = pixels.to(device), labels.to(device)
        loss_value, correct_count, batch_count = train_one_batch(model, optimizer, criterion, pixels, labels)
        train_loss += loss_value * batch_count
        train_correct += correct_count
        train_count += batch_count
    validation = evaluate(loaders["validation"])
    history.append({"stage": "finetune", "epoch": epoch, "train_loss": train_loss / train_count,
                    "train_accuracy": train_correct / train_count,
                    "validation_loss": validation["loss"], "validation_accuracy": validation["accuracy"]})
    key = (validation["accuracy"], -validation["loss"])
    if key > best_key:
        best_key, best_finetune_epoch = key, epoch
        best_tail = {name: parameter.detach().cpu().clone() for name, parameter in trainable.items()}
    print(f"finetune {epoch}/{FINETUNE_EPOCHS} · validation {validation['accuracy']:.1%}")
with torch.no_grad():
    for name, parameter in trainable.items():
        parameter.copy_(best_tail[name].to(device))
finetune_validation = evaluate(loaders["validation"])
finetune_test = evaluate(loaders["test"])
frozen_after = parameter_sha256((n, p) for n, p in model.named_parameters() if not p.requires_grad)
tail_after = parameter_sha256(model.vit.encoder.layer[-1].named_parameters())
assert frozen_before == frozen_after, "고정한 가중치가 바뀌었습니다."
assert tail_before != tail_after, "마지막 블록의 가중치가 바뀌지 않았습니다."
print(f"선택 epoch {best_finetune_epoch} · fine-tune test {finetune_test['accuracy']:.1%}")
FINETUNE_STAGE_COMPLETE = True
```

헤드 학습 셀과 반복 구조는 같습니다. 차이는 이번 optimizer가 마지막 블록·최종 정규화·헤드를 수정한다는 점입니다. 매 epoch마다 `model.train()`으로 시작하고, 학생이 만든 한 배치 함수로 업데이트한 뒤 `evaluate()`로 검증합니다.

평가 함수가 `model.eval()`로 바꾸기 때문에 다음 epoch를 시작할 때 다시 `train()`이 필요합니다. `requires_grad=False`는 가중치의 기울기 기록을 끄는 설정이고 `eval()`은 모델의 동작 모드를 바꾸는 설정이라는 차이를 떠올려 보세요.

`history`에는 단계 이름 `"finetune"`으로 기록합니다. 이번에도 `(검증 정확도, -검증 손실)`로 최적 시점을 선택합니다. `best_tail`에는 학습 대상 각각을 CPU로 복사해 그 시점의 값을 남깁니다.

반복 후 `torch.no_grad()` 안에서 `parameter.copy_(...)`로 선택한 시점의 가중치를 되돌립니다. 여기서는 새로운 학습이 아니라 이미 저장한 값을 복원하므로 미분 기록을 만들지 않습니다. 검증·테스트 평가도 그 복원된 모델로 합니다.

**마지막 두 검사:** `frozen_before == frozen_after`는 고정한 숫자들이 그대로인지, `tail_before != tail_after`는 마지막 블록의 숫자가 실제로 바뀌었는지 확인합니다. 가중치가 바뀌었다고 정확도 상승까지 보장되지는 않습니다. 전부 성공하면 `FINETUNE_STAGE_COMPLETE = True`가 됩니다.

## 19. 학습 중의 변화를 선 그래프로 읽기

**원본 셀 번호:** `34`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not FINETUNE_STAGE_COMPLETE:
    raise RuntimeError("실습 확인과 실제 학습을 완료한 뒤 결과를 확인하세요.")
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for stage, color in (("head", "#1a73e8"), ("finetune", "#188038")):
    rows = [row for row in history if row["stage"] == stage]
    epochs = [row["epoch"] for row in rows]
    axes[0].plot(epochs, [r["train_loss"] for r in rows], "o-", color=color, label=f"{stage}: train")
    axes[0].plot(epochs, [r["validation_loss"] for r in rows], "s--", color=color, label=f"{stage}: validation")
    axes[1].plot(epochs, [r["validation_accuracy"] for r in rows], "o-", color=color, label=stage)
axes[0].set(title="Loss by training stage", xlabel="Epoch within each stage", ylabel="Cross-entropy loss")
axes[1].set(title="Validation accuracy (100 images)", xlabel="Epoch within each stage", ylabel="Accuracy", ylim=(0, 1))
for axis in axes:
    axis.legend(fontsize=9)
    axis.grid(alpha=0.2)
    axis.set_xticks(range(1, max(HEAD_EPOCHS, FINETUNE_EPOCHS) + 1))
fig.savefig(OUTPUT_DIR / "learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
```

학습이 끝난 뒤 `history`의 기록으로 그림 두 개를 만듭니다. `plt.subplots(1, 2)`는 한 줄에 두 개의 그리기 공간을 준비합니다. `rows`는 `head` 또는 `finetune` 단계에 속한 기록만 고릅니다.

왼쪽은 손실입니다. 실선의 원 표시는 학습 데이터, 점선의 네모 표시는 검증 데이터입니다. 오른쪽은 검증 정확도이며 세로축을 0~1로 고정합니다. 색은 두 학습 단계를 구분합니다.

가로축은 전체를 이어 붙인 1~6이 아니라 **각 단계 안의 epoch 1~3**입니다. 같은 x 위치에 두 단계가 표시되어 있으므로 마지막 블록 학습의 epoch 1을 전체 학습의 첫 시점으로 읽지 않습니다.

`legend`는 선의 이름을, `grid`는 눈금을 넣습니다. `savefig`는 원본 실행에서 결과 폴더에 `learning_curves.png`를 저장하고 `show()`는 화면에 보여 줍니다. 이 해설에서는 해당 파일을 만들지 않습니다.

**그림을 보고 할 말:** 학습 손실만 낮아지고 검증 손실이 올라가는지 살펴봅니다. 세 번의 짧은 기록만으로 모든 추세를 단정할 수는 없습니다. 최적 모델은 그래프를 본 사람의 임의 판단이 아니라 앞서 정한 검증 기준으로 선택되었습니다.

## 20. 어떤 물체를 헷갈렸는지 비교하기

**원본 셀 번호:** `35`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not FINETUNE_STAGE_COMPLETE:
    raise RuntimeError("실습 확인과 실제 학습을 완료한 뒤 결과를 확인하세요.")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout="constrained")
maximum = max(np.max(head_test["confusion_matrix"]), np.max(finetune_test["confusion_matrix"]))
for axis, title, metrics in zip(axes, ("Head", "Fine-tune"), (head_test, finetune_test)):
    matrix = np.asarray(metrics["confusion_matrix"])
    image = axis.imshow(matrix, vmin=0, vmax=maximum, cmap="Blues")
    axis.set_xticks(range(len(classes)), classes, rotation=35, ha="right")
    axis.set_yticks(range(len(classes)), classes)
    axis.set(title=f"{title}: {metrics['accuracy']:.1%} ({metrics['count']} test images)",
             xlabel="Predicted class", ylabel="True class")
    for row, column in np.ndindex(matrix.shape):
        axis.text(column, row, str(matrix[row, column]), ha="center", va="center",
                  color="white" if matrix[row, column] > maximum / 2 else "black")
fig.colorbar(image, ax=axes, label="Image count", shrink=0.85)
fig.savefig(OUTPUT_DIR / "confusion_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"정확도 변화: {head_test['accuracy']:.1%} → {finetune_test['accuracy']:.1%}")
print(f"Macro F1 변화: {head_test['macro_f1']:.3f} → {finetune_test['macro_f1']:.3f}")
```

이번에는 `head_test`와 `finetune_test`의 혼동행렬을 나란히 보여 줍니다. 두 모델 모두 같은 최종 평가 이미지 200장을 사용합니다. 행은 실제 물체, 열은 예측한 물체입니다. 예를 들어 cup 행·bowl 열의 3은 “실제로 cup인 사진 3장을 bowl로 예측했다”는 뜻입니다.

`maximum`은 두 행렬에서 가장 큰 수입니다. 둘 다 `vmin=0, vmax=maximum`을 사용하므로 **같은 색이 같은 사진 수**를 나타냅니다. 각 그림을 따로 색 조절하면 같은 색이 서로 다른 수를 뜻할 수 있어 이렇게 맞춥니다.

`np.ndindex(matrix.shape)`는 각 행·열 위치를 순회합니다. 칸 안의 숫자는 해당 위치의 이미지 수이고, 진한 배경에서는 흰 글씨로 바꿔 읽기 쉽게 합니다. `colorbar`는 색과 사진 수의 관계를 표시합니다.

**실행 결과:** `confusion_comparison.png`를 저장하고 그림, 정확도 변화, Macro F1 변화를 보여 줍니다. 정확도 한 숫자만 올라도 특정 클래스에서는 더 나빠질 수 있으므로 대각선과 대각선 밖 숫자를 같이 봅니다. 0.80에서 0.81로 바뀌었다면 절대 변화는 1%p이지 1%의 상대 증가와 같은 표현이 아닙니다. 이것은 단위 설명용 숫자이며 이번 실행 결과는 출력에서 확인합니다.

### 작은 CPU 연습 · 혼동행렬에서 정확도와 F1 읽기

2종류, 10장의 가상 평가입니다. 행은 실제 cup·bowl, 열은 예측 cup·bowl 순서입니다. 오른쪽 위의 1은 cup을 bowl로 잘못 예측한 사진 한 장입니다.

**예상 출력:** 정확도 70.0%, cup F1 0.727, bowl F1 0.667, Macro F1 0.697. 같은 10장에서 8개를 맞혔다면 정확도는 10.0%p 올라갑니다. 실제 200장 평가 결과와 구분하세요.

In [7]:
matrix = [[4, 1], [2, 3]]
labels = ["cup", "bowl"]
total = sum(sum(row) for row in matrix)
correct = sum(matrix[i][i] for i in range(len(labels)))
accuracy = correct / total
print(f"정확도: {accuracy:.1%}")
f1_scores = []
for index, label in enumerate(labels):
    true_count = sum(matrix[index])
    predicted_count = sum(row[index] for row in matrix)
    denominator = true_count + predicted_count
    f1 = 2 * matrix[index][index] / denominator if denominator else 0.0
    f1_scores.append(f1)
    print(f"{label} F1: {f1:.3f}")
print(f"Macro F1: {sum(f1_scores) / len(f1_scores):.3f}")
print(f"7/10 → 8/10의 정확도 변화: {(8 / 10 - accuracy) * 100:+.1f}%p")

정확도: 70.0%
cup F1: 0.727
bowl F1: 0.667
Macro F1: 0.697
7/10 → 8/10의 정확도 변화: +10.0%p


## 21. 모델을 저장하고 다시 열어 같은 답이 나오는지 확인하기

**원본 셀 번호:** `37`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not FINETUNE_STAGE_COMPLETE:
    raise RuntimeError("실습 확인과 실제 학습을 완료한 뒤 결과를 확인하세요.")
MODEL_RELOADED = False
saved_model_dir = OUTPUT_DIR / "finetuned_model"
model.save_pretrained(saved_model_dir, safe_serialization=True)
processor.save_pretrained(saved_model_dir)
restored_processor = AutoImageProcessor.from_pretrained(saved_model_dir, local_files_only=True, use_fast=False)
restored = AutoModelForImageClassification.from_pretrained(
    saved_model_dir, local_files_only=True, use_safetensors=True, attn_implementation="eager",
).to(device).eval()
check_image = Image.fromarray(splits["test"]["images"][0]).convert("RGB")
check_input = processor(images=check_image, return_tensors="pt").to(device)
restored_input = restored_processor(images=check_image, return_tensors="pt").to(device)
model.eval()
with torch.inference_mode():
    before_save = model(**check_input).logits
    after_load = restored(**restored_input).logits
torch.testing.assert_close(before_save, after_load, rtol=1e-5, atol=1e-6)
assert restored.config.id2label == dict(enumerate(classes))
reload_max_abs_diff = float((before_save - after_load).abs().max())
print("저장·재로딩 검증 통과 · 최대 출력 차이:", reload_max_abs_diff)
MODEL_RELOADED = True
```

`model.save_pretrained(..., safe_serialization=True)`는 학습한 가중치를 Safetensors 형식으로 저장하고 모델 설정도 함께 남깁니다. `processor.save_pretrained()`는 전처리 설정을 저장합니다. 모델 가중치만 있고 입력 크기·정규화나 클래스 이름이 빠지면 이후 사용이 어려워집니다.

`restored_processor`와 `restored`는 방금 저장한 폴더에서 새로 만든 객체입니다. `local_files_only=True`로 로컬 파일만 사용하고 이번 저장 형식에 맞춰 `use_safetensors=True`를 지정합니다. `.to(device).eval()`은 새 모델도 GPU·평가 모드로 맞춥니다.

같은 테스트 사진 한 장을 원래 전처리와 다시 불러온 전처리에 각각 넣습니다. `model(**check_input)`의 `**`는 사전의 내용을 이름이 붙은 인자로 펼치는 표현입니다. 기울기 없이 두 모델의 logits를 계산하고 `assert_close`로 허용 오차 안에서 같은지 검사합니다.

`restored.config.id2label` 검사로 물체 이름도 보존되었는지 확인합니다. `reload_max_abs_diff`는 두 출력의 차이 중 절댓값이 가장 큰 숫자입니다.

**해석 범위:** 이 검사는 선택한 한 사진에서 저장·불러오기 전후의 출력과 라벨 설정을 확인합니다. 모든 이미지에서 완전히 같은 출력을 별도로 검증한 것은 아닙니다. 통과하면 `MODEL_RELOADED = True`가 되어 결과 보고서 저장을 진행할 수 있습니다.

## 22. 학습 기록과 사진별 예측을 CSV로 정리하기

**원본 셀 번호:** `39`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not FINETUNE_STAGE_COMPLETE:
    raise RuntimeError("실습 확인과 실제 학습을 완료한 뒤 결과를 확인하세요.")
if not MODEL_RELOADED:
    raise RuntimeError("모델 저장·재로딩 확인을 먼저 완료하세요.")
TABLES_SAVED = False
with (OUTPUT_DIR / "training.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(history[0]))
    writer.writeheader()
    writer.writerows(history)
with (OUTPUT_DIR / "predictions.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["sample_id", "true_label", "head_prediction", "finetune_prediction"])
    for sample_id, label, head_pred, fine_pred in zip(
        splits["test"]["ids"], splits["test"]["labels"], head_test["predictions"], finetune_test["predictions"],
    ):
        writer.writerow([sample_id, classes[int(label)], classes[head_pred], classes[fine_pred]])
def summary(metrics):
    return {key: value for key, value in metrics.items() if key != "predictions"}

TABLES_SAVED = True
```

GPU 학습과 모델 재로딩 확인을 마쳐야 파일을 저장합니다. `with ... open(...)`은 파일을 열어 작업하고 블록을 벗어나면 닫습니다. 이 셀은 원본 실행에서 파일을 쓰며 해설 노트북에서는 실행하지 않습니다.

`training.csv`에는 `history`를 기록합니다. `csv.DictWriter`는 사전의 키를 열 이름으로 사용합니다. `writeheader()`는 제목 행을, `writerows(history)`는 epoch별 기록을 씁니다. 정상 설정이면 head 3회와 finetune 3회로 데이터 행이 6개입니다.

`predictions.csv`는 각 테스트 사진을 한 행에 정리합니다. `zip(...)`은 같은 위치의 ID, 정답, 헤드 예측, 파인튜닝 예측을 한 묶음씩 가져옵니다. 정답과 예측의 숫자 번호를 `classes[...]`로 물체 이름에 연결하므로 사람이 읽을 수 있습니다. 정상 데이터라면 200개 예측 행이 있습니다.

`summary(metrics)`는 보고서용 사전에서 긴 개별 예측 목록만 제외합니다. 예측은 CSV에 이미 저장하므로 JSON 보고서에서는 손실·정확도·F1·개수·혼동행렬 중심으로 볼 수 있습니다. 마지막의 `TABLES_SAVED = True`는 두 표를 저장했다는 표시입니다.

**해석할 때:** CSV를 열어 같은 사진에서 어느 모델이 어떤 이름을 예측했는지 비교해 보세요. 행의 순서가 바뀌지 않도록 검증·테스트 DataLoader에서 `shuffle=False`로 두었던 이유가 여기서도 드러납니다.

## 23. 실행 조건과 검증 결과를 보고서에 모으기

**원본 셀 번호:** `40`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not FINETUNE_STAGE_COMPLETE:
    raise RuntimeError("실습 확인과 실제 학습을 완료한 뒤 결과를 확인하세요.")
if not MODEL_RELOADED:
    raise RuntimeError("모델 저장·재로딩 확인을 먼저 완료하세요.")
REPORT_READY = False
if not TABLES_SAVED:
    raise RuntimeError("결과 표 저장을 먼저 완료하세요.")
report = {
    "status": "completed", "platform": "gpu", "accelerator_verified": True,
    "device": torch.cuda.get_device_name(0), "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "download_manifest_sha256": sha256_file(MODEL_DIR / "download_manifest.json"),
    "dataset_manifest_sha256": sha256_file(manifest_file), "dataset_sha256": manifest["dataset_sha256"],
    "classes": classes, "split_counts": {k: len(v["labels"]) for k, v in splits.items()},
    "versions": {name: version(name) for name in ("torch", "transformers", "huggingface-hub")},
    "config": {"seed": SEED, "batch_size": BATCH_SIZE, "head_epochs": HEAD_EPOCHS,
               "finetune_epochs": FINETUNE_EPOCHS, "head_lr": HEAD_LR, "finetune_lr": FINETUNE_LR,
               "optimizer": "Adam", "checkpoint_selection": "validation_accuracy_then_loss"},
    "trainable_parameters": trainable_count,
    "stages": {"head": {"selected_epoch": best_head_epoch, "validation": summary(head_validation), "test": summary(head_test)},
               "finetune": {"selected_epoch": best_finetune_epoch, "validation": summary(finetune_validation), "test": summary(finetune_test)}},
    "verification": {"frozen_parameters_unchanged": frozen_before == frozen_after,
                     "last_block_changed": tail_before != tail_after, "reload_max_abs_diff": reload_max_abs_diff,
                     "frozen_before": frozen_before, "frozen_after": frozen_after,
                     "last_block_before": tail_before, "last_block_after": tail_after},
    "checkpoint": {"path": "finetuned_model", "format": "safetensors",
                   "sha256": sha256_file(saved_model_dir / "model.safetensors")},
    "elapsed_seconds": time.perf_counter() - started,
}
REPORT_READY = True
```

앞 단계가 전부 끝났는지 확인한 후 `report` 사전을 만듭니다. 이 셀은 아직 `report.json` 파일을 쓰지 않습니다. 다음 셀에서 파일 목록의 SHA-256까지 붙여 저장합니다.

| 사전 항목 | 읽을 내용 |
|---|---|
| `status`, `platform`, `device` | 완료 상태, GPU 사용, 실제 GPU 이름 |
| `model_id`, `model_revision`, manifest SHA-256 | 어떤 모델·데이터 기록으로 실행했는지 |
| `classes`, `split_counts`, `versions` | 클래스 순서, 사진 수, 라이브러리 버전 |
| `config` | seed, 배치, epoch, 학습률, optimizer, 모델 선택 기준 |
| `stages` | 각 단계에서 고른 epoch와 검증·테스트 지표 |
| `verification` | 고정 가중치 유지, 마지막 블록 변화, 재로딩 차이 |
| `checkpoint` | 모델 저장 위치, 형식, 가중치 파일 SHA-256 |
| `elapsed_seconds` | 준비 단계의 시작 시각부터 여기까지 경과 시간 |

`{k: len(v["labels"]) for ...}` 같은 사전 내포는 분할별 사진 수를 한 줄로 모으는 표현입니다. `summary()`로 개별 예측 목록만 덜어낸 지표를 기록합니다.

SHA-256은 파일의 같은 내용을 확인하는 데 쓰고, 정확도·F1은 모델의 동작을 평가하는 데 씁니다. 같은 파일이라는 증거와 모델이 얼마나 잘 맞히는지는 서로 다른 질문입니다. 또 `elapsed_seconds`에는 데이터 준비와 평가·저장에 걸린 시간도 포함되므로 GPU 학습 반복문만의 속도로 해석하지 않습니다.

마지막 `REPORT_READY = True`는 현재 실행의 보고서 자료를 모았다는 뜻입니다. 완료 문자열이 사전에 있다고 해서 실패한 앞 단계를 건너뛸 수는 없습니다.

## 24. 결과 파일을 확인하고 완료 보고서 저장하기

**원본 셀 번호:** `41`

**원본 코드 — 읽기 전용**

```python
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not FINETUNE_STAGE_COMPLETE:
    raise RuntimeError("실습 확인과 실제 학습을 완료한 뒤 결과를 확인하세요.")
if not REPORT_READY:
    raise RuntimeError("현재 실행의 보고서 준비가 완료되지 않았습니다.")
artifact_names = [
    "training.csv", "predictions.csv", "learning_curves.png", "confusion_comparison.png",
    "finetuned_model/config.json", "finetuned_model/preprocessor_config.json",
    "finetuned_model/model.safetensors",
]
report["artifacts"] = artifact_names
report["artifact_sha256"] = {
    name: sha256_file(OUTPUT_DIR / name) for name in artifact_names
}
(OUTPUT_DIR / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("HF_GPU_FINETUNING_COMPLETE", OUTPUT_DIR / "report.json")
```

원본의 마지막 코드 셀입니다. 학습 완료와 보고서 준비 표시를 다시 확인한 다음, 전달할 결과 7개를 `artifact_names`에 적습니다. CSV 2개, PNG 그림 2개, 모델 설정·전처리 설정·Safetensors 가중치 3개입니다.

`artifact_sha256`은 각 결과 파일을 읽어 내용의 SHA-256을 기록합니다. 파일이 빠져 있으면 읽는 단계에서 오류가 나므로 불완전한 결과를 완료 보고서로 저장하지 않습니다.

`json.dumps(report, ensure_ascii=False, indent=2)`는 Python 사전을 들여쓰기 있는 JSON 글로 바꿉니다. `ensure_ascii=False`는 한글을 그대로 보이게 합니다. `write_text(..., encoding="utf-8")`에서 비로소 `report.json` 파일을 씁니다.

**정상 마지막 출력:** `HF_GPU_FINETUNING_COMPLETE` 다음에 보고서 경로가 나옵니다. 출력 문구만으로 모든 결과를 알 수는 없으므로 `report.json`과 결과 파일도 함께 확인합니다. 파일 목록에는 보고서 자체를 넣지 않았습니다. 자기 내용의 SHA-256을 자기 안에 넣어 비교하려 하면 내용이 다시 바뀌기 때문입니다.

**원본 실습을 마치면:** Colab 세션을 종료하기 전에 결과 폴더와 모델을 내려받습니다. 원격 세션에만 남겨 둔 파일은 수업 종료 후에도 보존된다고 가정하지 않습니다. 실행 절차는 [HF·Colab 실습 안내](../README.md)를 따릅니다.

## 용어를 자기 말로 설명하기

| 용어 | 이 실습에서의 뜻 |
|---|---|
| 파라미터·가중치 | 모델이 계산에 사용하는 숫자. 학습할 때 선택한 부분을 바꿉니다. |
| 헤드·classifier | 특징을 물체 종류별 점수로 바꾸는 마지막 분류층입니다. |
| logits | softmax 전의 점수. 정답 번호와 함께 CrossEntropyLoss에 넣습니다. |
| 배치 | 한 번에 처리하는 사진 묶음입니다. 마지막 묶음의 크기는 작을 수 있습니다. |
| epoch | 학습 사진 전체를 한 번 읽는 단위입니다. |
| 손실 | 모델의 예측과 정답이 얼마나 어긋나는지 학습에 사용하는 값입니다. |
| 기울기·backward | 손실에 따라 각 가중치를 어느 방향으로 바꿀지 계산합니다. |
| optimizer·step | 계산한 기울기와 학습률을 사용해 가중치를 실제로 바꿉니다. |
| requires_grad | 해당 가중치의 학습용 기울기를 계산할지 정하는 설정입니다. |
| train·eval | 학습할 때와 평가할 때 모델의 동작 모드를 선택합니다. |
| 검증·최종 평가 | 검증은 모델 선택에, 남겨 둔 최종 평가는 선택한 모델 평가에 씁니다. |
| 체크포인트 | 다시 불러올 수 있도록 저장한 특정 시점의 모델입니다. |
| SHA-256 | 파일이나 숫자 묶음의 내용이 같은지 비교하는 문자열입니다. 성능 지표는 아닙니다. |

## 이해했는지 확인하기

1. 사진 500장을 16장씩 읽을 때 왜 배치별 평균 손실을 그냥 32로 나누면 안 될까요?
2. `model.eval()`과 `requires_grad=False`가 하는 일은 어떻게 다른가요?
3. 학생 함수에서 `backward()`만 하고 `step()`을 빼면 가중치는 바뀔까요?
4. 검증 정확도가 같은 두 epoch 중 하나를 고르는 기준은 무엇인가요?
5. `head → last_block → head` 순서로 선택했을 때 마지막에 965개가 되어야 하는 이유는 무엇인가요?
6. 혼동행렬의 cup 행·bowl 열은 무엇을 세나요?
7. 모델을 저장할 때 전처리 설정과 클래스 이름도 함께 남기는 이유는 무엇인가요?

<details>
<summary>생각한 뒤 개념 확인하기</summary>

1. 마지막 배치는 4장이라 16장 배치와 같은 비중을 주면 사진별 평균이 달라집니다.
2. eval은 모델의 동작 모드이고 requires_grad는 가중치의 기울기 계산 여부입니다.
3. backward는 기울기를 계산합니다. 실제 값 변경에는 optimizer의 step이 필요합니다.
4. 검증 손실이 낮은 시점을 고릅니다. 둘 다 같으면 먼저 고른 시점을 유지합니다.
5. 범위를 바꿀 때 이전에 켠 선택을 지우고 classifier만 다시 켜야 하기 때문입니다.
6. 실제 cup인데 bowl로 예측한 사진 수입니다.
7. 같은 입력 변환을 하고 점수의 번호를 같은 물체 이름으로 읽기 위해서입니다.

</details>

## 원본 실습으로 돌아가기

- [02 · GPU 파인튜닝 원본](../notebooks/02_gpu_finetuning.ipynb): 두 빈칸을 작성하고 검사를 통과한 뒤 실제 학습합니다.
- [01 · GPU 추론 원본](../notebooks/01_gpu_inference.ipynb): 전처리·logits·라벨 연결을 먼저 확인합니다.
- [00 · 모델과 데이터 준비 원본](../notebooks/00_hf_download_and_data.ipynb): Codespaces 환경과 파일 준비를 확인합니다.
- [AI 코딩 실습 안내](../ai-coding-workshop.md): 조건을 바꾸어 본 과정과 결과를 기록합니다.
- [HF·Colab 실행 안내](../README.md): 원격 실행, 결과 회수, 세션 종료 절차를 확인합니다.

이 해설의 작은 CPU 연습 결과는 개념 확인용입니다. 실제 모델의 정확도·학습 시간·GPU 사용 결과는 원본을 실행해서 생성한 `report.json`과 CSV·그림으로 확인하세요.